# DQN vs REINFORCE on CartPole-v1

This notebook recreates two figures:

1. **DQN vs REINFORCE (with baseline)**: learning curves over 5 seeds.
2. **DQN ablation**: DQN with vs without a target network.

Training uses the functions in `dqn_starter.py` and `policy_gradient_starter.py` (starter code supplied by the ST962 module at the University of Warwick). This notebook adds the experiment runs, result caching, plotting and summary statistics.

**Runtime:** training is CPU-heavy (3 configurations × 5 seeds × 1,000 episodes) and can take an hour or more on a laptop. The returns are saved to `results/cartpole_returns.npz` after the first run, so later runs only redraw the plots. Set `RERUN = True` to retrain from scratch.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from dqn_starter import run_multiple_seeds as run_dqn_seeds
from policy_gradient_starter import run_multiple_seeds as run_pg_seeds, train_reinforce

ENV_NAME = "CartPole-v1"
N_SEEDS = 5
N_EPISODES = 1000
WINDOW = 20             # moving-average window for the plots
SOLVED = 475            # CartPole-v1 "solved" threshold
RERUN = False           # True = retrain even if saved results exist

RESULTS_FILE = Path("results/cartpole_returns.npz")
FIG_DIR = Path("figures")
RESULTS_FILE.parent.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

## 1. Train (or load saved results)

Each run returns an array of shape `(N_SEEDS, N_EPISODES)`: the undiscounted return (number of steps the pole stayed up) for every episode of every seed.

In [ ]:
if RESULTS_FILE.exists() and not RERUN:
    data = np.load(RESULTS_FILE)
    results = {key: data[key] for key in data.files}
    print(f"Loaded saved results from {RESULTS_FILE}")
else:
    results = {}

    print("=== DQN with target network ===")
    results["dqn"] = run_dqn_seeds(ENV_NAME, n_seeds=N_SEEDS, n_episodes=N_EPISODES,
                                   use_target_network=True)

    print("\n=== DQN without target network (ablation) ===")
    results["dqn_no_target"] = run_dqn_seeds(ENV_NAME, n_seeds=N_SEEDS, n_episodes=N_EPISODES,
                                             use_target_network=False)

    print("\n=== REINFORCE with baseline ===")
    results["reinforce"] = run_pg_seeds(train_reinforce, n_seeds=N_SEEDS, env_name=ENV_NAME,
                                        n_episodes=N_EPISODES, use_baseline=True)

    np.savez(RESULTS_FILE, **results)
    print(f"\nSaved results to {RESULTS_FILE}")

for key, arr in results.items():
    print(f"{key:15s} shape={arr.shape}")

## 2. Plotting helper

For each method: average the returns across seeds, smooth the mean and standard deviation with a 20-episode moving average, and shade ±1 standard deviation. A dashed line marks the solved threshold of 475.

In [ ]:
def moving_average(x, window):
    return np.convolve(x, np.ones(window) / window, mode="valid")


def plot_curves(curves, title, ax=None):
    # curves: list of (label, returns array of shape (seeds, episodes), colour)
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))

    for label, returns, colour in curves:
        mean = moving_average(returns.mean(axis=0), WINDOW)
        std = moving_average(returns.std(axis=0), WINDOW)
        x = np.arange(len(mean))
        ax.plot(x, mean, color=colour, linewidth=1.5, label=label)
        ax.fill_between(x, mean - std, mean + std, color=colour, alpha=0.2)

    ax.axhline(SOLVED, color="green", linestyle="--", linewidth=1.5,
               label=f"Solved threshold ({SOLVED})")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Return")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    return ax

## 3. DQN vs REINFORCE

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_curves(
    [
        ("DQN (with target network)", results["dqn"], "steelblue"),
        ("REINFORCE (with baseline)", results["reinforce"], "tomato"),
    ],
    title=(f"DQN vs REINFORCE — {ENV_NAME}\n"
           f"{N_SEEDS} seeds, ±1 std shading, {WINDOW}-episode moving average"),
    ax=ax,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "dqn_vs_reinforce.png", dpi=140, bbox_inches="tight")
plt.show()

## 4. DQN ablation: target network vs no target network

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_curves(
    [
        ("DQN (with target network)", results["dqn"], "steelblue"),
        ("DQN (no target network)", results["dqn_no_target"], "orange"),
    ],
    title=(f"DQN Ablation: Target Network vs No Target Network\n"
           f"{ENV_NAME}, {N_SEEDS} seeds, ±1 std shading"),
    ax=ax,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "dqn_target_network_ablation.png", dpi=140, bbox_inches="tight")
plt.show()

## 5. Summary

- **Final (smoothed):** the last point of each curve, i.e. the seed-averaged return over the final 20 episodes.
- **Last 100 episodes:** mean return over episodes 901–1,000, averaged over seeds, ± the standard deviation across seeds.
- **First episode ≥ 400:** the first episode where the smoothed curve reaches 400 (a measure of learning speed).

In [ ]:
labels = {
    "dqn": "DQN (with target network)",
    "dqn_no_target": "DQN (no target network)",
    "reinforce": "REINFORCE (with baseline)",
}

print(f"{'Method':28s} {'Final (smoothed)':>17s} {'Last 100 eps':>18s} {'First ep >= 400':>16s}")
print("-" * 82)
for key, label in labels.items():
    returns = results[key]
    smoothed = moving_average(returns.mean(axis=0), WINDOW)
    per_seed_last100 = returns[:, -100:].mean(axis=1)
    reached = np.flatnonzero(smoothed >= 400)
    first_400 = str(reached[0]) if reached.size else "never"
    print(f"{label:28s} {smoothed[-1]:17.1f} "
          f"{per_seed_last100.mean():10.1f} ± {per_seed_last100.std():5.1f} {first_400:>16s}")